In [ ]:
import pandas as pd
df = pd.read_csv("Hospital_Cleaned_Data.csv")
# Convert appointment date and time
df["Appointment_DateTime"] = pd.to_datetime(df["Appointment_DateTime"], errors="coerce", utc=True)
# Extract time slot
df["Time_Slot"] = df["Appointment_DateTime"].dt.strftime("%H:00")
# Group by specialty and time slot
result = df.groupby(["Specialty", "Time_Slot"]).agg(
    Total_Appointments=("Appointment_ID", "count"),
    Attended=("Appointment_Status",
              lambda x: x.isin(["ARRIVED", "COMPLETED"]).sum()),
    Cancelled=("Appointment_Status",
               lambda x: (x == "CANCELLED").sum()),
    No_Show=("Appointment_Status",
             lambda x: (x == "NO-SHOW").sum())
).reset_index()
# Calculate rates
result["Attendance_Rate"] = (result["Attended"] / result["Total_Appointments"] * 100).round(2)
result["Cancellation_Rate"] = (result["Cancelled"] / result["Total_Appointments"] * 100).round(2)
result["No_Show_Rate"] = (result["No_Show"] / result["Total_Appointments"] * 100).round(2)

print(result)

      Specialty Time_Slot  Total_Appointments  Attended  Cancelled  No_Show  \
0    CARDIOLOGY     00:00                 106        30         29       31   
1    CARDIOLOGY     01:00                 102        31         33       25   
2    CARDIOLOGY     02:00                 116        31         38       30   
3    CARDIOLOGY     03:00                 106        34         23       38   
4    CARDIOLOGY     04:00                 111        25         35       30   
..          ...       ...                 ...       ...        ...      ...   
187  PEDIATRICS     19:00                 103        23         29       36   
188  PEDIATRICS     20:00                 105        28         31       35   
189  PEDIATRICS     21:00                  94        22         33       21   
190  PEDIATRICS     22:00                  94        17         37       25   
191  PEDIATRICS     23:00                 116        33         34       30   

     Attendance_Rate  Cancellation_Rate  No_Show_Ra

In [ ]:
#2. Measure registration-to-triage, triage-to-provider, and arrival-to-discharge durations
# Convert timestamps to UTC only when timezone information exists
def convert_time(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NaT
    time = pd.to_datetime(value, errors="coerce", utc=False)
    if pd.isna(time) or time.tzinfo is None:
        return pd.NaT
    return time.tz_convert("UTC")
for column in ["Arrival", "Triage", "Discharge"]:
    df[column] = df[column].apply(convert_time)
# Calculate arrival-to-discharge duration in minutes
df["Duration_Minutes"] = (df["Discharge"] - df["Arrival"]).dt.total_seconds() / 60

# Check for missing timestamps
df["Issue"] = ""
df.loc[
    df["Arrival"].isna() | df["Triage"].isna() | df["Discharge"].isna(),"Issue"] = "Missing timestamp or unknown timezone"
# Check whether events occurred in the correct order
invalid = (
    (df["Triage"] < df["Arrival"])
    | (df["Discharge"] < df["Arrival"])
    | (df["Discharge"] < df["Triage"])
)
df.loc[invalid, "Issue"] = "Invalid event sequence"
# Exclude cancelled visits from the duration calculation
cancelled = df["Cancelled_Encounter"].fillna(False).astype(bool)
df.loc[(df["Issue"] != "") | invalid | cancelled,"Duration_Minutes"] = float("nan")
# Store Scenario 2 results in a variable
duration_results = df[[
    "Encounter_ID",
    "Arrival",
    "Triage",
    "Discharge",
    "Duration_Minutes",
    "Issue"
]].copy()

print(duration_results)

      Encounter_ID                   Arrival                    Triage  \
0       ENC-000001 2026-01-25 02:20:00+00:00 2026-01-25 03:10:00+00:00   
1       ENC-000002                       NaT                       NaT   
2       ENC-000003                       NaT                       NaT   
3       ENC-000004                       NaT                       NaT   
4       ENC-000005                       NaT                       NaT   
...            ...                       ...                       ...   
99995   ENC-099996 2026-06-15 10:04:00+00:00 2026-06-15 10:54:00+00:00   
99996   ENC-099997                       NaT                       NaT   
99997   ENC-099998                       NaT                       NaT   
99998   ENC-099999                       NaT                       NaT   
99999   ENC-100000                       NaT                       NaT   

                      Discharge  Duration_Minutes  \
0     2026-01-25 07:39:00+00:00             319.0   
1    

In [ ]:
#3. Build daily admission, discharge, transfer, and occupancy summaries by ward.
#  Convert timestamps without assuming unknown timezones
def convert_time(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NaT
    time = pd.to_datetime(value, errors="coerce")

    if pd.isna(time) or time.tzinfo is None:
        return pd.NaT

    return time.tz_convert("UTC")


for column in ["Admission", "Discharge", "Transfer"]:
    df[column] = df[column].apply(convert_time)

# Keep valid operational records
df = df[
    (df["Include_in_Operations"] == True)
    & (df["Test_Record"] == False)
    & (df["Cancelled_Encounter"] == False)
    & (df["Event_Order_Issue"] == False)
    & (df["Discharge_Before_Admission"] == False)
].copy()

# Create daily admission, discharge and transfer events
events = []
for column, event_name in [
    ("Admission", "Admissions"),
    ("Discharge", "Discharges"),
    ("Transfer", "Transfers")]:
    temp = df.dropna(subset=[column, "Ward"])[["Encounter_ID", "Ward", column]].copy()
    temp["Date"] = temp[column].apply(lambda x: x.date())
    temp["Event"] = event_name
    events.append(
        temp[["Encounter_ID", "Ward", "Date", "Event"]]
    )

events = pd.concat(events, ignore_index=True)

# Avoid counting the same encounter event twice
events = events.drop_duplicates(subset=["Encounter_ID", "Ward", "Date", "Event"])

# Count events by date and ward
daily_events = events.groupby(["Date", "Ward", "Event"])["Encounter_ID"].nunique().unstack(fill_value=0)

# Calculate estimated occupied encounters at the end of each day
valid_admissions = df.dropna(
    subset=["Admission", "Ward"]
).copy()

if valid_admissions.empty:
    ward_daily_summary = pd.DataFrame(
        columns=[
            "Date", "Ward", "Admissions",
            "Discharges", "Transfers", "Occupied_Encounters"
        ]
    )

else:
    first_day = valid_admissions["Admission"].min().normalize()
    last_day = valid_admissions["Admission"].max().normalize()

    if valid_admissions["Discharge"].notna().any():
        last_day = max(
            last_day,
            valid_admissions["Discharge"].max().normalize()
        )

    occupancy = []

    for day in pd.date_range(
        start=first_day,
        end=last_day,
        freq="D"
    ):
        next_day = day + pd.Timedelta(days=1)

        active = valid_admissions[
            (valid_admissions["Admission"] < next_day)
            & (
                (valid_admissions["Discharge"] >= next_day)
                | (
                    valid_admissions["Discharge"].isna()
                    & valid_admissions["Status"].str.upper().isin(
                        ["ADMITTED", "IN PROGRESS"]
                    )
                )
            )
        ]

        counts = active.groupby("Ward")["Encounter_ID"].nunique()

        for ward, count in counts.items():
            occupancy.append({
                "Date": day.date(),
                "Ward": ward,
                "Occupied_Encounters": count
            })

    daily_occupancy = pd.DataFrame(occupancy)

    # Combine daily events and occupancy
    ward_daily_summary = daily_events.reset_index().merge(
        daily_occupancy,
        on=["Date", "Ward"],
        how="outer"
    )

    # Fill missing event counts with zero
    for column in ["Admissions", "Discharges", "Transfers",
                   "Occupied_Encounters"]:
        if column not in ward_daily_summary.columns:
            ward_daily_summary[column] = 0
        else:
            ward_daily_summary[column] = (
                ward_daily_summary[column].fillna(0).astype(int)
            )

    ward_daily_summary = ward_daily_summary.sort_values(
        ["Date", "Ward"]
    ).reset_index(drop=True)

# Display the result
print(ward_daily_summary.head(10))


         Date             Ward  Admissions  Discharges  Transfers  \
0  2026-01-01  CARDIOLOGY WARD           5           2          3   
1  2026-01-01   EMERGENCY WARD           1           1          1   
2  2026-01-01     GENERAL WARD           2           2          2   
3  2026-01-01              ICU           4           4          4   
4  2026-01-01   PEDIATRIC WARD           4           2          2   
5  2026-01-01    SURGICAL WARD           4           2          3   
6  2026-01-02  CARDIOLOGY WARD           6           6          7   
7  2026-01-02   EMERGENCY WARD          10           7          9   
8  2026-01-02     GENERAL WARD           6           5          6   
9  2026-01-02              ICU           6           4          5   

   Occupied_Encounters  
0                    3  
1                    0  
2                    0  
3                    0  
4                    2  
5                    2  
6                    3  
7                    3  
8              

In [ ]:
#4. Calculate bed occupancy and turnover using documented available-bed and time-window definitions.
#  Keep valid operational records
df = df[
    (df["Include_in_Operations"] == True)
    & (df["Test_Record"] == False)
    & (df["Cancelled_Encounter"] == False)
].copy()

#  Standardize bed information
df["Bed_ID"] = df["Bed_ID"].astype("string").str.strip()
df["Bed_Inventory_Status"] = (
    df["Bed_Inventory_Status"].astype("string").str.strip().str.lower()
)

# Ignore records without a bed ID or status
beds = df.dropna(
    subset=["Bed_ID", "Bed_Inventory_Status", "Facility", "Ward"]
).copy()

beds = beds[
    (beds["Bed_ID"] != "")
    & (beds["Bed_Inventory_Status"] != "")
]

#  A bed with conflicting statuses cannot be classified reliably
bed_status_counts = beds.groupby(
    ["Facility", "Bed_ID"]
)["Bed_Inventory_Status"].nunique()

conflicting_beds = bed_status_counts[bed_status_counts > 1].index

beds["Status_Conflict"] = beds.set_index(
    ["Facility", "Bed_ID"]
).index.isin(conflicting_beds)

# Keep only beds with one consistent status
valid_beds = beds[~beds["Status_Conflict"]].drop_duplicates(
    subset=["Facility", "Bed_ID"]
)

#Count occupied, available and out-of-service beds
bed_summary = valid_beds.groupby(
    ["Facility", "Bed_Inventory_Status"]
)["Bed_ID"].nunique().unstack(fill_value=0)

for status in ["occupied", "available", "out of service"]:
    if status not in bed_summary.columns:
        bed_summary[status] = 0

bed_summary = bed_summary.rename(columns={
    "occupied": "Occupied_Beds",
    "available": "Available_Beds",
    "out of service": "Out_of_Service_Beds"
})

# Calculate snapshot occupancy rate
# Out-of-service beds are excluded from the denominator
bed_summary["Counted_Beds"] = (
    bed_summary["Occupied_Beds"] + bed_summary["Available_Beds"]
)

bed_summary["Occupancy_Rate_Percent"] = (
    bed_summary["Occupied_Beds"]
    .div(bed_summary["Counted_Beds"].replace(0, pd.NA))
    * 100
)

# Turnover cannot be calculated reliably without a
# documented reporting window and bed-availability definition
bed_summary["Bed_Turnover"] = float("nan")

bed_summary["Turnover_Issue"] = (
    "Not calculated: reporting window and bed-capacity "
    "definition are not documented"
)

# Store the result; do not export a CSV yet
bed_summary = bed_summary.reset_index()

print(bed_summary.head(10))

Bed_Inventory_Status          Facility  Available_Beds  cleaning  \
0                     CENTRAL HOSPITAL               4         4   
1                        EAST HOSPITAL               3         4   
2                       NORTH-HOSPITAL               6         8   
3                       SOUTH-HOSPITAL               7         3   
4                        WEST HOSPITAL               4         5   

Bed_Inventory_Status  Occupied_Beds  Out_of_Service_Beds  Counted_Beds  \
0                                20                    3            24   
1                                19                    6            22   
2                                22                    4            28   
3                                21                    2            28   
4                                19                    2            23   

Bed_Inventory_Status  Occupancy_Rate_Percent  Bed_Turnover  \
0                                  83.333333           NaN   
1                     

In [ ]:
#5. Measure laboratory turnaround time from specimen collection to verified result.
# Check for the required specimen collection timestamp
collection_column = "Specimen_Collection_DateTime"

if collection_column not in df.columns:
    # This column is absent from the current dataset.
    # Do not invent collection timestamps.
    df[collection_column] = pd.NaT


#  Convert timestamps to UTC only when a timezone is present
def convert_time(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NaT

    try:
        timestamp = pd.Timestamp(value)

        # Unknown timezone: leave unresolved (TC05)
        if timestamp.tzinfo is None:
            return pd.NaT

        return timestamp.tz_convert("UTC")

    except (ValueError, TypeError):
        return pd.NaT


# Force both columns to a consistent datetime dtype
df[collection_column] = pd.to_datetime(
    df[collection_column].apply(convert_time),
    utc=True,
    errors="coerce"
)

df["Lab_Result_DateTime"] = pd.to_datetime(
    df["Lab_Result_DateTime"].apply(convert_time),
    utc=True,
    errors="coerce"
)


#  Keep only verified lab results
verified = df[
    df["Lab_Result_Status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("verified")
].copy()


# Calculate turnaround time in minutes
verified["Lab_TAT_Minutes"] = (
    verified["Lab_Result_DateTime"]
    - verified[collection_column]
).dt.total_seconds() / 60


#  Flag data quality issues
verified["TAT_Issue"] = ""

missing_collection = verified[collection_column].isna()
missing_result = verified["Lab_Result_DateTime"].isna()
negative_tat = verified["Lab_TAT_Minutes"] < 0

verified.loc[
    missing_collection,
    "TAT_Issue"
] = "Missing or unresolved specimen collection timestamp"

verified.loc[
    missing_result,
    "TAT_Issue"
] = verified.loc[missing_result, "TAT_Issue"].apply(
    lambda issue: (
        issue + "; " if issue else ""
    ) + "Missing or unresolved verified-result timestamp"
)

verified.loc[
    negative_tat,
    "TAT_Issue"
] = "Result timestamp precedes specimen collection"

# Do not retain negative turnaround times
verified.loc[negative_tat, "Lab_TAT_Minutes"] = float("nan")


#  Store the results; do not export a CSV yet
lab_tat_results = verified[
    [
        "Encounter_ID",
        "Lab_Order_ID",
        "Lab_Result_Status",
        collection_column,
        "Lab_Result_DateTime",
        "Lab_TAT_Minutes",
        "TAT_Issue"
    ]
].copy()

print(lab_tat_results.head(10))
print("\nRows with TAT issues:", (lab_tat_results["TAT_Issue"] != "").sum())


   Encounter_ID Lab_Order_ID Lab_Result_Status Specimen_Collection_DateTime  \
0    ENC-000001  LAB-0000001          VERIFIED                          NaT   
5    ENC-000006  LAB-0000006          VERIFIED                          NaT   
13   ENC-000014  LAB-0000014          VERIFIED                          NaT   
19   ENC-000020  LAB-0000020          VERIFIED                          NaT   
22   ENC-000023  LAB-0000023          VERIFIED                          NaT   
34   ENC-000035  LAB-0000035          VERIFIED                          NaT   
36   ENC-000037  LAB-0000037          VERIFIED                          NaT   
37   ENC-000038  LAB-0000038          VERIFIED                          NaT   
41   ENC-000042  LAB-0000042          VERIFIED                          NaT   
42   ENC-000043  LAB-0000043          VERIFIED                          NaT   

         Lab_Result_DateTime  Lab_TAT_Minutes  \
0  2026-01-25 09:23:00+00:00              NaN   
5  2026-08-21 00:00:00+00:00    

In [6]:
#6. Summarize test volume, result delays, and workload by laboratory section.
# 2. Keep valid operational records
def is_true(value):
    return str(value).strip().lower() in ["true", "1", "yes"]


if "Include_in_Operations" in df.columns:
    df = df[df["Include_in_Operations"].apply(is_true)].copy()

if "Test_Record" in df.columns:
    df = df[~df["Test_Record"].apply(is_true)].copy()

if "Cancelled_Encounter" in df.columns:
    df = df[~df["Cancelled_Encounter"].apply(is_true)].copy()

if "Orphan_Lab_Result" in df.columns:
    df = df[~df["Orphan_Lab_Result"].apply(is_true)].copy()


# 3. Standardize test names and order IDs
df["Lab_Test"] = (
    df["Lab_Test"].astype("string").str.strip().str.upper()
)

df["Lab_Order_ID"] = (
    df["Lab_Order_ID"].astype("string").str.strip().str.upper()
)

df["Lab_Result_Status"] = (
    df["Lab_Result_Status"].astype("string").str.strip().str.upper()
)

df["Lab_Test"] = df["Lab_Test"].fillna("UNKNOWN TEST TYPE")


# 4. Remove exact duplicate lab event records
df = df.drop_duplicates(
    subset=[
        "Lab_Order_ID",
        "Lab_Test",
        "Lab_Result_Status",
        "Lab_Result_DateTime"
    ]
).copy()


# 5. Parse result timestamps without assuming a missing timezone
def convert_time(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NaT

    try:
        timestamp = pd.Timestamp(value)

        # Unknown timezone: leave unresolved
        if timestamp.tzinfo is None:
            return pd.NaT

        return timestamp.tz_convert("UTC")

    except (ValueError, TypeError):
        return pd.NaT


df["Lab_Result_DateTime"] = pd.to_datetime(
    df["Lab_Result_DateTime"].apply(convert_time),
    utc=True,
    errors="coerce"
)


# 6. Create indicators for result status and missing timestamps
df["Is_Verified"] = df["Lab_Result_Status"].eq("VERIFIED")
df["Is_Pending"] = df["Lab_Result_Status"].eq("PENDING")
df["Is_Preliminary"] = df["Lab_Result_Status"].eq("PRELIMINARY")
df["Is_Corrected"] = df["Lab_Result_Status"].eq("CORRECTED")
df["Missing_Result_Time"] = df["Lab_Result_DateTime"].isna()


# 7. Summarize test volume and workload by available test type
lab_section_summary = df.groupby("Lab_Test").agg(
    Test_Volume=("Lab_Order_ID", "nunique"),
    Verified_Results=("Is_Verified", "sum"),
    Pending_Results=("Is_Pending", "sum"),
    Preliminary_Results=("Is_Preliminary", "sum"),
    Corrected_Results=("Is_Corrected", "sum"),
    Missing_Result_Timestamps=("Missing_Result_Time", "sum")
).reset_index()


# 8. Add a clear limitation for result delays
lab_section_summary["Delay_Measure_Issue"] = (
    "Elapsed delay unavailable: specimen collection/order timestamp "
    "and laboratory section are not provided"
)


# 9. Store the result; do not export a CSV yet
lab_section_summary = lab_section_summary.sort_values(
    by="Test_Volume",
    ascending=False
).reset_index(drop=True)

print(lab_section_summary.head(10))


               Lab_Test  Test_Volume  Verified_Results  Pending_Results  \
0                   WBC         8120              2665             1369   
1  COMPLETE BLOOD COUNT         7924              2657             1295   
2               GLUCOSE         7894              2632             1299   
3           LIPID PANEL         7883              2519             1358   
4            CREATININE         7873              2579             1365   
5            HEMOGLOBIN         7865              2657             1353   
6             POTASSIUM         7826              2648             1258   
7                   ALT         7745              2589             1292   
8                   AST         7734              2584             1306   
9                SODIUM         7715              2615             1291   

   Preliminary_Results  Corrected_Results  Missing_Result_Timestamps  \
0                 1332               1389                       7264   
1                 1285        

In [7]:
#7. Calculate average length of stay and stratify by ward or service line.
# 2. Helper function for reading true/false flags
def is_true(value):
    return str(value).strip().lower() in ["true", "1", "yes"]


# 3. Keep valid operational records
if "Include_in_Operations" in df.columns:
    df = df[df["Include_in_Operations"].apply(is_true)].copy()

if "Test_Record" in df.columns:
    df = df[~df["Test_Record"].apply(is_true)].copy()

if "Cancelled_Encounter" in df.columns:
    df = df[~df["Cancelled_Encounter"].apply(is_true)].copy()

if "Event_Order_Issue" in df.columns:
    df = df[~df["Event_Order_Issue"].apply(is_true)].copy()

if "Discharge_Before_Admission" in df.columns:
    df = df[~df["Discharge_Before_Admission"].apply(is_true)].copy()


# 4. Convert timestamps to UTC only if timezone is present
def convert_time(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NaT

    try:
        timestamp = pd.Timestamp(value)

        # Do not assume a timezone if it is missing
        if timestamp.tzinfo is None:
            return pd.NaT

        return timestamp.tz_convert("UTC")

    except (ValueError, TypeError):
        return pd.NaT


df["Admission"] = pd.to_datetime(
    df["Admission"].apply(convert_time),
    utc=True,
    errors="coerce"
)

df["Discharge"] = pd.to_datetime(
    df["Discharge"].apply(convert_time),
    utc=True,
    errors="coerce"
)


# 5. Remove rows missing essential information
df["Encounter_ID"] = (
    df["Encounter_ID"].astype("string").str.strip().str.upper()
)

df["Ward"] = df["Ward"].astype("string").str.strip()

df = df[
    df["Encounter_ID"].notna()
    & (df["Encounter_ID"] != "")
    & df["Ward"].notna()
    & (df["Ward"] != "")
    & df["Admission"].notna()
    & df["Discharge"].notna()
].copy()


# 6. Calculate length of stay for each row
df["LOS_Hours"] = (
    df["Discharge"] - df["Admission"]
).dt.total_seconds() / 3600

# Remove negative or zero stays
df = df[df["LOS_Hours"] > 0].copy()


# 7. Avoid counting the same encounter more than once
# Exclude encounters assigned to multiple wards because
# ward-specific stay durations cannot be established safely
ward_counts = df.groupby("Encounter_ID")["Ward"].nunique()

single_ward_encounters = ward_counts[ward_counts == 1].index

df = df[df["Encounter_ID"].isin(single_ward_encounters)].copy()

# Collapse repeated records to one encounter-level LOS
encounter_los = (
    df.groupby("Encounter_ID")
    .agg(
        Ward=("Ward", "first"),
        Admission=("Admission", "min"),
        Discharge=("Discharge", "max")
    )
    .reset_index()
)

encounter_los["LOS_Hours"] = (
    encounter_los["Discharge"] - encounter_los["Admission"]
).dt.total_seconds() / 3600

encounter_los = encounter_los[
    encounter_los["LOS_Hours"] > 0
].copy()


# 8. Summarize average LOS by ward
los_by_ward = (
    encounter_los.groupby("Ward")
    .agg(
        Completed_Encounters=("Encounter_ID", "nunique"),
        Average_LOS_Hours=("LOS_Hours", "mean"),
        Median_LOS_Hours=("LOS_Hours", "median"),
        Minimum_LOS_Hours=("LOS_Hours", "min"),
        Maximum_LOS_Hours=("LOS_Hours", "max")
    )
    .reset_index()
)

los_by_ward["Average_LOS_Days"] = (
    los_by_ward["Average_LOS_Hours"] / 24
)

los_by_ward = los_by_ward.sort_values(
    "Average_LOS_Hours",
    ascending=False
).reset_index(drop=True)


# 9. Store results; do not export a CSV yet
print(los_by_ward.head(10))


              Ward  Completed_Encounters  Average_LOS_Hours  Median_LOS_Hours  \
0  CARDIOLOGY WARD                  1426           6.504897          6.625000   
1     GENERAL WARD                  1441           6.443500          6.450000   
2   EMERGENCY WARD                  1471           6.420893          6.550000   
3              ICU                  1414           6.412400          6.475000   
4   PEDIATRIC WARD                  1412           6.381858          6.483333   
5    SURGICAL WARD                  1473           6.377325          6.466667   

   Minimum_LOS_Hours  Maximum_LOS_Hours  Average_LOS_Days  
0           1.300000          11.050000          0.271037  
1           1.016667          11.000000          0.268479  
2           1.083333          11.083333          0.267537  
3           0.816667          10.966667          0.267183  
4           1.016667          11.216667          0.265911  
5           0.616667          11.133333          0.265722  


In [8]:
#8. Measure emergency department arrivals and waiting-time percentiles by hour.
# Convert only timestamps that contain a known timezone
def convert_time(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NaT

    try:
        timestamp = pd.Timestamp(value)

        if timestamp.tzinfo is None:
            return pd.NaT

        return timestamp.tz_convert("UTC")

    except (ValueError, TypeError):
        return pd.NaT


# Keep emergency department records
df["Dept"] = df["Dept"].astype("string").str.strip().str.upper()

df = df[df["Dept"].isin(["ER", "EMERGENCY", "EMERGENCY DEPARTMENT"])].copy()


# Exclude records that should not count in operational metrics
def is_true(value):
    return str(value).strip().lower() in ["true", "1", "yes"]


if "Include_in_Operations" in df.columns:
    df = df[df["Include_in_Operations"].apply(is_true)].copy()

if "Test_Record" in df.columns:
    df = df[~df["Test_Record"].apply(is_true)].copy()

if "Cancelled_Encounter" in df.columns:
    df = df[~df["Cancelled_Encounter"].apply(is_true)].copy()

if "Status" in df.columns:
    df = df[
        ~df["Status"].astype("string").str.contains(
            "cancel", case=False, na=False
        )
    ].copy()


# Clean encounter IDs
df["Encounter_ID"] = (
    df["Encounter_ID"].astype("string").str.strip().str.upper()
)

df = df[
    df["Encounter_ID"].notna()
    & (df["Encounter_ID"] != "")
].copy()


# Convert arrival and triage timestamps consistently to UTC
df["Arrival"] = pd.to_datetime(
    df["Arrival"].apply(convert_time),
    utc=True,
    errors="coerce"
)

df["Triage"] = pd.to_datetime(
    df["Triage"].apply(convert_time),
    utc=True,
    errors="coerce"
)


# Remove rows without a usable arrival timestamp
df = df[df["Arrival"].notna()].copy()


# Remove exact duplicate encounter events
df = df.drop_duplicates(
    subset=["Encounter_ID", "Arrival", "Triage"]
).copy()


# Calculate arrival-to-triage waiting time in minutes
df["Waiting_Minutes"] = (
    df["Triage"] - df["Arrival"]
).dt.total_seconds() / 60


# Negative waits are invalid and must not enter percentiles
df.loc[df["Waiting_Minutes"] < 0, "Waiting_Minutes"] = float("nan")


# Group arrivals by UTC hour
df["Arrival_Hour"] = df["Arrival"].dt.hour


# Summarize arrival volume and waiting-time percentiles
ed_hourly_summary = (
    df.groupby("Arrival_Hour")
    .agg(
        ED_Arrivals=("Encounter_ID", "nunique"),
        Valid_Wait_Count=("Waiting_Minutes", "count"),
        P50_Wait_Minutes=("Waiting_Minutes", "median"),
        P90_Wait_Minutes=(
            "Waiting_Minutes",
            lambda values: values.quantile(0.90)
        ),
        P95_Wait_Minutes=(
            "Waiting_Minutes",
            lambda values: values.quantile(0.95)
        )
    )
    .reset_index()
    .sort_values("Arrival_Hour")
)

print(ed_hourly_summary)

    Arrival_Hour  ED_Arrivals  Valid_Wait_Count  P50_Wait_Minutes  \
0              0           40                39              60.0   
1              1           40                40              58.0   
2              2           43                43              50.0   
3              3           46                46              52.0   
4              4           54                54              48.5   
5              5           50                50              45.5   
6              6           37                37              51.0   
7              7           52                50              59.5   
8              8           52                51              48.0   
9              9           53                53              45.0   
10            10           36                36              52.0   
11            11           36                36              55.5   
12            12           51                51              57.0   
13            13           47     

In [9]:
#9. Build staffing coverage summaries by unit and shift where roster data is available.
import re
# 2. Clean basic text columns
for col in ["Facility", "Ward", "Dept", "Staff_Shift"]:
    df[col] = df[col].astype("string").str.strip()

df["Staff_Shift"] = df["Staff_Shift"].str.upper()

# Use Ward as the unit; use Dept if Ward is missing
df["Unit"] = df["Ward"].replace("", pd.NA)
df["Unit"] = df["Unit"].fillna(df["Dept"].replace("", pd.NA))

# 3. Convert shift timestamps to UTC
df["Shift_Start_DT"] = pd.to_datetime(
    df["Shift_Start"], errors="coerce", utc=True
)
df["Shift_End_DT"] = pd.to_datetime(
    df["Shift_End"], errors="coerce", utc=True
)

# 4. Calculate shift duration
df["Shift_Duration_Hours"] = (
    df["Shift_End_DT"] - df["Shift_Start_DT"]
).dt.total_seconds() / 3600

# 5. Validate each shift record
df["Shift_Issue"] = ""

missing_details = (
    df["Facility"].isna()
    | df["Facility"].eq("")
    | df["Unit"].isna()
    | df["Unit"].eq("")
    | df["Staff_Shift"].isna()
    | df["Staff_Shift"].eq("")
)

df.loc[
    missing_details, "Shift_Issue"
] = "Missing facility, unit, or shift label"

missing_time = (
    df["Shift_Start_DT"].isna()
    | df["Shift_End_DT"].isna()
)

df.loc[
    missing_time, "Shift_Issue"
] = "Missing or invalid shift timestamp"

invalid_duration = df["Shift_Duration_Hours"].le(0).fillna(False)

df.loc[
    invalid_duration, "Shift_Issue"
] = "End time is not after start time"

long_duration = df["Shift_Duration_Hours"].gt(16).fillna(False)

df.loc[
    long_duration, "Shift_Issue"
] = "Duration exceeds 16 hours; verify schedule"

# 6. Check labels such as 22:00-06:00 against timestamps
def check_shift_label(row):
    label = row["Staff_Shift"]
    start = row["Shift_Start_DT"]
    end = row["Shift_End_DT"]

    pattern = r"(\d{1,2}):(\d{2})\s*[-–]\s*(\d{1,2}):(\d{2})"
    match = re.fullmatch(pattern, str(label).strip())

    # Labels such as DAY, NIGHT, and EVENING
    # cannot be checked against exact clock times
    if not match:
        return None

    # Missing timestamps are handled separately
    if pd.isna(start) or pd.isna(end):
        return None

    sh, sm, eh, em = map(int, match.groups())

    if sh > 23 or eh > 23 or sm > 59 or em > 59:
        return False

    expected_start = (sh, sm)
    expected_end = (eh, em)

    actual_start = (start.hour, start.minute)
    actual_end = (end.hour, end.minute)

    expected_minutes = (
        (eh * 60 + em) - (sh * 60 + sm)
    ) % 1440

    actual_minutes = (
        end - start
    ).total_seconds() / 60

    return (
        actual_start == expected_start
        and actual_end == expected_end
        and actual_minutes == expected_minutes
    )


label_checks = df.apply(check_shift_label, axis=1)

explicit_label = df["Staff_Shift"].str.fullmatch(
    r"\d{1,2}:\d{2}\s*[-–]\s*\d{1,2}:\d{2}",
    na=False
)

timestamps_available = (
    df["Shift_Start_DT"].notna()
    & df["Shift_End_DT"].notna()
)

label_mismatch = (
    explicit_label
    & timestamps_available
    & label_checks.eq(False)
)

df.loc[
    label_mismatch, "Shift_Issue"
] = "Explicit shift label conflicts with timestamps"

# 7. Separate valid records and exceptions
staffing_shift_exceptions = df[
    df["Shift_Issue"] != ""
].copy()

valid = df[
    df["Shift_Issue"] == ""
].copy()

# 8. Confirm that every row is accounted for
print("========== ROW ACCOUNTING ==========")
print("Original rows:", len(df))
print("Valid rows:", len(valid))
print("Exception rows:", len(staffing_shift_exceptions))
print("Rows accounted for:", len(valid) + len(staffing_shift_exceptions))

assert len(valid) + len(staffing_shift_exceptions) == len(df)

# 9. Remove repeated records for the same shift
shift_keys = [
    "Facility",
    "Unit",
    "Staff_Shift",
    "Shift_Start_DT",
    "Shift_End_DT"
]

valid_unique = valid.drop_duplicates(
    subset=shift_keys
).copy()

valid_unique["Shift_Date_UTC"] = (
    valid_unique["Shift_Start_DT"].dt.date
)

# 10. Create the shift summary
staffing_coverage_summary = (
    valid_unique
    .groupby(
        [
            "Facility",
            "Unit",
            "Shift_Date_UTC",
            "Staff_Shift"
        ],
        dropna=False
    )
    .agg(
        Recorded_Shift_Entries=("Shift_Start_DT", "size"),
        Average_Duration_Hours=("Shift_Duration_Hours", "mean"),
        Minimum_Duration_Hours=("Shift_Duration_Hours", "min"),
        Maximum_Duration_Hours=("Shift_Duration_Hours", "max")
    )
    .reset_index()
)

# 11. Summarize the exceptions
exception_summary = (
    staffing_shift_exceptions["Shift_Issue"]
    .value_counts()
    .rename_axis("Shift_Issue")
    .reset_index(name="Record_Count")
)

# 12. Display results
print("\n========== STAFFING SHIFT SUMMARY ==========")
print(staffing_coverage_summary.to_string(index=False))

print("\n========== EXCEPTION SUMMARY ==========")
print(exception_summary.to_string(index=False))

print("\n========== EXCEPTION SAMPLE ==========")
print(
    staffing_shift_exceptions[
        [
            "Facility",
            "Unit",
            "Staff_Shift",
            "Shift_Start",
            "Shift_End",
            "Shift_Duration_Hours",
            "Shift_Issue"
        ]
    ].head(10).to_string(index=False)
)


========== ROW ACCOUNTING ==========
Original rows: 1082
Valid rows: 225
Exception rows: 857
Rows accounted for: 1082

========== STAFFING SHIFT SUMMARY ==========
        Facility            Unit Shift_Date_UTC Staff_Shift  Recorded_Shift_Entries  Average_Duration_Hours  Minimum_Duration_Hours  Maximum_Duration_Hours
CENTRAL HOSPITAL CARDIOLOGY WARD     2026-10-26         DAY                       1                8.000000                     8.0                     8.0
CENTRAL HOSPITAL CARDIOLOGY WARD     2026-10-26     EVENING                       1               15.000000                    15.0                    15.0
CENTRAL HOSPITAL CARDIOLOGY WARD     2026-10-26       NIGHT                       3                5.333333                     1.0                     8.0
CENTRAL HOSPITAL  EMERGENCY WARD     2026-10-26         DAY                       2               12.000000                     9.0                    15.0
CENTRAL HOSPITAL  EMERGENCY WARD     2026-10-26     EVEN

In [10]:
#10. Calculate procedure cancellation rates and reason-code distributions.
# 2. Clean procedure-related columns
columns = [
    "Facility",
    "Specialty",
    "Procedure_Code",
    "Procedure_Status",
    "Procedure_Cancellation_Reason"
]

for col in columns:
    df[col] = df[col].astype("string").str.strip()

# 3. Identify procedure records
has_procedure = (
    df["Procedure_Code"].notna()
    | df["Procedure_Status"].notna()
    | df["Procedure_Cancellation_Reason"].notna()
)

procedures = df[has_procedure].copy()

# Exclude records explicitly marked as test records
if "Test_Record" in procedures.columns:
    test_values = (
        procedures["Test_Record"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    is_test = test_values.isin(
        ["true", "yes", "y", "1", "test"]
    )

    procedures = procedures[~is_test].copy()

# 4. Standardize procedure statuses
procedures["Status_Clean"] = (
    procedures["Procedure_Status"]
    .str.lower()
    .str.strip()
    .str.replace(r"[\s_-]+", " ", regex=True)
)

status_mapping = {
    "cancelled": "Cancelled",
    "canceled": "Cancelled",
    "complete": "Completed",
    "completed": "Completed",
    "in progress": "In Progress",
    "scheduled": "Scheduled",
    "pending": "Pending",
    "held": "Held"
}

procedures["Status_Clean"] = (
    procedures["Status_Clean"]
    .map(status_mapping)
    .fillna(procedures["Status_Clean"])
)

# Blank statuses remain missing
procedures.loc[
    procedures["Procedure_Status"].isna()
    | procedures["Procedure_Status"].eq(""),
    "Status_Clean"
] = pd.NA

# 5. Separate cancelled and known-status procedures
known_status = procedures[
    procedures["Status_Clean"].notna()
].copy()

cancelled = known_status[
    known_status["Status_Clean"] == "Cancelled"
].copy()

# 6. Calculate the overall cancellation rate
total_procedures = len(known_status)
cancelled_count = len(cancelled)

if total_procedures > 0:
    cancellation_rate = (
        cancelled_count / total_procedures
    ) * 100
else:
    cancellation_rate = float("nan")

procedure_cancellation_summary = pd.DataFrame({
    "Total_Procedures_With_Known_Status": [total_procedures],
    "Cancelled_Procedures": [cancelled_count],
    "Cancellation_Rate_Percent": [round(cancellation_rate, 2)]
})

# 7. Calculate cancellation rates by facility and specialty
procedure_cancellation_by_service = (
    known_status
    .groupby(["Facility", "Specialty"], dropna=False)
    .agg(
        Total_Procedures=("Status_Clean", "size"),
        Cancelled_Procedures=(
            "Status_Clean",
            lambda s: (s == "Cancelled").sum()
        )
    )
    .reset_index()
)

procedure_cancellation_by_service[
    "Cancellation_Rate_Percent"
] = (
    procedure_cancellation_by_service["Cancelled_Procedures"]
    / procedure_cancellation_by_service["Total_Procedures"]
    * 100
).round(2)

# 8. Analyze cancellation reasons
cancelled["Reason_Clean"] = (
    cancelled["Procedure_Cancellation_Reason"]
    .replace("", pd.NA)
    .fillna("Missing reason")
)

procedure_reason_summary = (
    cancelled
    .groupby("Reason_Clean", dropna=False)
    .size()
    .reset_index(name="Cancelled_Procedure_Count")
    .sort_values(
        "Cancelled_Procedure_Count",
        ascending=False
    )
)

if cancelled_count > 0:
    procedure_reason_summary["Percentage_of_Cancellations"] = (
        procedure_reason_summary["Cancelled_Procedure_Count"]
        / cancelled_count
        * 100
    ).round(2)
else:
    procedure_reason_summary["Percentage_of_Cancellations"] = 0.0

# 9. Flag records needing review
procedure_cancellation_exceptions = procedures[
    procedures["Status_Clean"].isna()
    | (
        (procedures["Status_Clean"] == "Cancelled")
        & procedures["Procedure_Cancellation_Reason"].isna()
    )
    | (
        (procedures["Status_Clean"] == "Cancelled")
        & procedures["Procedure_Cancellation_Reason"].eq("")
    )
].copy()

procedure_cancellation_exceptions["Issue"] = ""

procedure_cancellation_exceptions.loc[
    procedure_cancellation_exceptions["Status_Clean"].isna(),
    "Issue"
] = "Missing procedure status"

missing_reason = (
    (procedure_cancellation_exceptions["Status_Clean"] == "Cancelled")
    & (
        procedure_cancellation_exceptions[
            "Procedure_Cancellation_Reason"
        ].isna()
        | procedure_cancellation_exceptions[
            "Procedure_Cancellation_Reason"
        ].eq("")
    )
)

procedure_cancellation_exceptions.loc[
    missing_reason, "Issue"
] = "Cancelled procedure has no recorded reason"

# 10. Display results
print("========== OVERALL CANCELLATION SUMMARY ==========")
print(procedure_cancellation_summary.to_string(index=False))

print("\n========== CANCELLATION RATE BY FACILITY AND SPECIALTY ==========")
print(procedure_cancellation_by_service.to_string(index=False))

print("\n========== CANCELLATION REASON DISTRIBUTION ==========")
print(procedure_reason_summary.to_string(index=False))

print("\n========== EXCEPTION SUMMARY ==========")
print(
    procedure_cancellation_exceptions["Issue"]
    .value_counts()
    .to_string()
)

print("\nRecords needing review:",
      len(procedure_cancellation_exceptions))


========== OVERALL CANCELLATION SUMMARY ==========
 Total_Procedures_With_Known_Status  Cancelled_Procedures  Cancellation_Rate_Percent
                               1082                   362                      33.46

========== CANCELLATION RATE BY FACILITY AND SPECIALTY ==========
        Facility          Specialty  Total_Procedures  Cancelled_Procedures  Cancellation_Rate_Percent
CENTRAL HOSPITAL         CARDIOLOGY                21                     5                      23.81
CENTRAL HOSPITAL      CRITICAL CARE                23                     8                      34.78
CENTRAL HOSPITAL EMERGENCY MEDICINE                26                    11                      42.31
CENTRAL HOSPITAL    GENERAL SURGERY                33                     9                      27.27
CENTRAL HOSPITAL  INTERNAL MEDICINE                21                     6                      28.57
CENTRAL HOSPITAL        ORTHOPEDICS                29                     9                   

In [11]:
#11. Reconcile billed, paid, and outstanding amounts by payer and service category.
# 2. Convert billing columns to numeric
amount_columns = [
    "Billed_Amount",
    "Paid_Amount",
    "Outstanding_Amount"
]

for col in amount_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 3. Exclude test records
test_records = (
    df["Test_Record"]
    .astype("string")
    .str.strip()
    .str.lower()
    .isin(["true", "yes", "y", "1", "test"])
)

billing = df.loc[~test_records].copy()

# 4. Calculate expected outstanding and reconciliation difference
billing["Expected_Outstanding"] = (
    billing["Billed_Amount"] - billing["Paid_Amount"]
).round(2)

billing["Reconciliation_Difference"] = (
    billing["Outstanding_Amount"]
    - billing["Expected_Outstanding"]
).round(2)

# 5. Flag records requiring review
billing["Billing_Exception"] = ""

for index, row in billing.iterrows():
    issues = []

    if pd.isna(row["Payer"]) or str(row["Payer"]).strip() == "":
        issues.append("Missing payer")

    if pd.isna(row["Specialty"]) or str(row["Specialty"]).strip() == "":
        issues.append("Missing service category")

    if any(pd.isna(row[col]) for col in amount_columns):
        issues.append("Missing or invalid amount")

    if any(
        pd.notna(row[col]) and row[col] < 0
        for col in amount_columns
    ):
        issues.append("Negative amount - review")

    if (
        pd.notna(row["Reconciliation_Difference"])
        and abs(row["Reconciliation_Difference"]) > 0.01
    ):
        issues.append("Outstanding amount mismatch")

    billing.at[index, "Billing_Exception"] = "; ".join(issues)

# 6. Create summary by payer and specialty
billing_reconciliation_summary = (
    billing.groupby(["Payer", "Specialty"], dropna=False)
    .agg(
        Record_Count=("Encounter_ID", "size"),
        Billed_Total=("Billed_Amount", lambda s: s.sum(min_count=1)),
        Paid_Total=("Paid_Amount", lambda s: s.sum(min_count=1)),
        Recorded_Outstanding=(
            "Outstanding_Amount",
            lambda s: s.sum(min_count=1)
        ),
        Expected_Outstanding=(
            "Expected_Outstanding",
            lambda s: s.sum(min_count=1)
        )
    )
    .reset_index()
)

billing_reconciliation_summary["Reconciliation_Difference"] = (
    billing_reconciliation_summary["Recorded_Outstanding"]
    - billing_reconciliation_summary["Expected_Outstanding"]
).round(2)

# 7. Create exceptions table
billing_reconciliation_exceptions = billing.loc[
    billing["Billing_Exception"] != ""
].copy()

# 8. Print results
print("\n========== BILLING RECONCILIATION SUMMARY ==========")
print(billing_reconciliation_summary.to_string(index=False))

print("\n========== RECONCILIATION EXCEPTIONS ==========")
print(
    billing_reconciliation_exceptions[
        [
            "Encounter_ID",
            "Payer",
            "Specialty",
            "Billed_Amount",
            "Paid_Amount",
            "Outstanding_Amount",
            "Expected_Outstanding",
            "Reconciliation_Difference",
            "Billing_Exception"
        ]
    ].to_string(index=False)
)

# 9. Print quality checks
mismatch_count = (
    billing["Reconciliation_Difference"].abs() > 0.01
).fillna(False).sum()

print("\n========== QUALITY CHECKS ==========")
print("Original rows:", len(df))
print("Rows included in reconciliation:", len(billing))
print("Rows in summary:", billing_reconciliation_summary["Record_Count"].sum())
print("Rows needing review:", len(billing_reconciliation_exceptions))
print("Rows with outstanding mismatches:", mismatch_count)

print("\nException reasons:")
print(
    billing_reconciliation_exceptions["Billing_Exception"]
    .str.split("; ")
    .explode()
    .value_counts()
    .to_string()
)



========== BILLING RECONCILIATION SUMMARY ==========
            Payer          Specialty  Record_Count  Billed_Total  Paid_Total  Recorded_Outstanding  Expected_Outstanding  Reconciliation_Difference
        CORPORATE         CARDIOLOGY            18    1333013.50   722910.49             610103.01             610103.01                       0.00
        CORPORATE      CRITICAL CARE            17    1290586.07   565688.32             724897.75             724897.75                       0.00
        CORPORATE EMERGENCY MEDICINE            23    1839755.84   966527.01             873228.83             873228.83                       0.00
        CORPORATE    GENERAL SURGERY            24    1938074.32  1050281.73             887792.59             887792.59                       0.00
        CORPORATE  INTERNAL MEDICINE            19    1450152.30   810128.62             640023.68             640023.68                       0.00
        CORPORATE        ORTHOPEDICS            23    1473

In [12]:
import os
print("File location:", os.path.abspath("Hospital_Cleaned_Data.csv"))
print("Dataset shape:", df.shape)

print("\nTest_Record values:")
print(df["Test_Record"].value_counts(dropna=False))

print("\nInclude_in_Operations values:")
print(df["Include_in_Operations"].value_counts(dropna=False))

print("\nBilling_Issue values:")
print(df["Billing_Issue"].value_counts(dropna=False))

File location: c:\Users\subhashis\OneDrive\Documents\GitHub\Hospital-Operations-Patient-Flow-Analysis\Hospital_Cleaned_Data.csv
Dataset shape: (1082, 73)

Test_Record values:
Test_Record
False    1082
Name: count, dtype: int64

Include_in_Operations values:
Include_in_Operations
True    1082
Name: count, dtype: int64

Billing_Issue values:
Billing_Issue
False    991
True      91
Name: count, dtype: int64


In [13]:
df = pd.read_csv("Hospital_Cleaned_Data.csv")

In [14]:
#11. Reconcile billed, paid, and outstanding amounts by payer and service category.
# Convert billing columns to numeric
amounts = ["Billed_Amount", "Paid_Amount", "Outstanding_Amount"]
df[amounts] = df[amounts].apply(pd.to_numeric, errors="coerce")

# Reconciliation calculations
df["Expected_Outstanding"] = df["Billed_Amount"] - df["Paid_Amount"]
df["Reconciliation_Difference"] = (
    df["Outstanding_Amount"] - df["Expected_Outstanding"]
)

# Exception flags
df["Missing_Payer"] = (
    df["Payer"].isna() | df["Payer"].astype(str).str.strip().eq("")
)
df["Negative_Amount"] = df[amounts].lt(0).any(axis=1)
df["Outstanding_Mismatch"] = (
    df["Reconciliation_Difference"].abs() > 0.01
)

df["Needs_Review"] = df[
    ["Missing_Payer", "Negative_Amount", "Outstanding_Mismatch"]
].any(axis=1)

# Summary by payer and specialty
summary = df.groupby(
    ["Payer", "Specialty"], dropna=False
).agg(
    Record_Count=("Encounter_ID", "size"),
    Total_Billed=("Billed_Amount", "sum"),
    Total_Paid=("Paid_Amount", "sum"),
    Recorded_Outstanding=("Outstanding_Amount", "sum"),
    Expected_Outstanding=("Expected_Outstanding", "sum"),
    Mismatch_Records=("Outstanding_Mismatch", "sum")
).reset_index()

summary["Difference"] = (
    summary["Recorded_Outstanding"] - summary["Expected_Outstanding"]
)

# Final report
print("BILLING RECONCILIATION SUMMARY")
print(summary.to_string(index=False, float_format="%.2f"))

print("\nDATA QUALITY REPORT")
print("Total records:", len(df))
print("Records needing review:", int(df["Needs_Review"].sum()))
print("Missing payer:", int(df["Missing_Payer"].sum()))
print("Negative amount - review:", int(df["Negative_Amount"].sum()))
print("Outstanding mismatches:", int(df["Outstanding_Mismatch"].sum()))
print("Payer-specialty groups:", len(summary))


BILLING RECONCILIATION SUMMARY
            Payer          Specialty  Record_Count  Total_Billed   Total_Paid  Recorded_Outstanding  Expected_Outstanding  Mismatch_Records  Difference
        CORPORATE         CARDIOLOGY          1819  135166945.96  73247410.67           60987415.05           61919535.29                11  -932120.24
        CORPORATE      CRITICAL CARE          1790  136594609.51  77025976.26           58616113.75           59568633.25                10  -952519.50
        CORPORATE EMERGENCY MEDICINE          1809  135119992.16  71877482.26           62583546.60           63242509.90                13  -658963.30
        CORPORATE    GENERAL SURGERY          1862  138921650.39  75371383.16           63055917.05           63550267.23                 9  -494350.18
        CORPORATE  INTERNAL MEDICINE          1697  129958838.27  71751107.48           57805938.79           58207730.79                 7  -401792.00
        CORPORATE        ORTHOPEDICS          1828  13911

In [15]:
#12. Create capacity forecasts from historical arrivals and occupancy with assumptions documented.
# 1. Convert timestamps
df["Arrival"] = pd.to_datetime(df["Arrival"], errors="coerce")
df["Admission"] = pd.to_datetime(df["Admission"], errors="coerce")
df["Discharge"] = pd.to_datetime(df["Discharge"], errors="coerce")

# 2. Forecast daily arrivals
daily_arrivals = (
    df.dropna(subset=["Arrival"])
      .groupby(df["Arrival"].dt.date)
      .size()
)

average_arrivals = daily_arrivals.mean()

print("Average Daily Arrivals:", round(average_arrivals, 2))
print("Forecast Arrivals Per Day for Next 7 Days:",
      round(average_arrivals, 2))

# 3. Estimate daily occupancy
valid_admissions = df.dropna(subset=["Admission"])

if not valid_admissions.empty:
    start_date = valid_admissions["Admission"].min().normalize()
    end_date = valid_admissions["Admission"].max().normalize()

    daily_occupancy = []

    for day in pd.date_range(start_date, end_date):
        next_day = day + pd.Timedelta(days=1)

        occupied = (
            (df["Admission"] < next_day)
            & df["Admission"].notna()
            & (
                df["Discharge"].isna()
                | (df["Discharge"] >= next_day)
            )
        )

        daily_occupancy.append(occupied.sum())

    average_occupancy = sum(daily_occupancy) / len(daily_occupancy)

    print("\nAverage Estimated Daily Occupancy:",
          round(average_occupancy, 2))
    print("Baseline Occupancy Estimate for Next 7 Days:",
          round(average_occupancy, 2), "per day")

else:
    print("\nOccupancy forecast unavailable: no valid admission timestamps.")

# 4. Document assumptions
print("\nAssumptions:")
print("1. Historical daily arrivals are used as the baseline forecast.")
print("2. Demand is assumed to remain stable for the next 7 days.")
print("3. Occupancy is estimated from admission and discharge timestamps.")
print("4. Missing discharge times may overestimate occupancy.")
print("5. Each row is treated as one encounter; duplicate records may inflate estimates.")
print("6. Occupancy estimates require validation against actual bed records.")
print("7. These are baseline estimates, not validated capacity predictions.")

Average Daily Arrivals: 74.07
Forecast Arrivals Per Day for Next 7 Days: 74.07

Average Estimated Daily Occupancy: 13.65
Baseline Occupancy Estimate for Next 7 Days: 13.65 per day

Assumptions:
1. Historical daily arrivals are used as the baseline forecast.
2. Demand is assumed to remain stable for the next 7 days.
3. Occupancy is estimated from admission and discharge timestamps.
4. Missing discharge times may overestimate occupancy.
5. Each row is treated as one encounter; duplicate records may inflate estimates.
6. Occupancy estimates require validation against actual bed records.
7. These are baseline estimates, not validated capacity predictions.


In [16]:
#13. Track data completeness for key timestamps, encounter IDs, and department assignments.
key_columns = [
    "Encounter_ID",
    "Arrival",
    "Triage",
    "Discharge",
    "Admission",
    "Dept"
]

completeness_report = pd.DataFrame({
    "Total_Rows": df[key_columns].count() + df[key_columns].isna().sum(),
    "Missing_Count": df[key_columns].isna().sum(),
    "Completeness_Percent": (
        df[key_columns].notna().mean() * 100
    ).round(2)
})

print("Data Completeness Report")
print(completeness_report)

print("\nTotal records:", len(df))
print("Records missing Encounter_ID:", df["Encounter_ID"].isna().sum())
print("Records missing Dept:", df["Dept"].isna().sum())

timestamp_columns = [
    "Arrival", "Triage", "Discharge", "Admission"
]

print(
    "\nRecords missing at least one key timestamp:",
    df[timestamp_columns].isna().any(axis=1).sum()
)

Data Completeness Report
              Total_Rows  Missing_Count  Completeness_Percent
Encounter_ID      100000              0                 100.0
Arrival           100000          80000                  20.0
Triage            100000          80000                  20.0
Discharge         100000          80000                  20.0
Admission         100000          80000                  20.0
Dept              100000              0                 100.0

Total records: 100000
Records missing Encounter_ID: 0
Records missing Dept: 0

Records missing at least one key timestamp: 80000


In [17]:
#14. Identify outlier durations and sudden changes in volume for operational review.
# 1. Convert timestamps
for col in ["Arrival", "Triage", "Discharge"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# 2. Calculate durations in minutes
df["Arrival_to_Triage_Min"] = (
    df["Triage"] - df["Arrival"]
).dt.total_seconds() / 60

df["Arrival_to_Discharge_Hours"] = (
    df["Discharge"] - df["Arrival"]
).dt.total_seconds() / 3600

# 3. Identify outlier durations using the IQR method
duration_columns = [
    "Arrival_to_Triage_Min",
    "Arrival_to_Discharge_Hours"
]

for col in duration_columns:
    values = df[col].dropna()
    values = values[values >= 0]

    if len(values) == 0:
        print(f"\n{col}: No valid durations available.")
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    upper_limit = q3 + 1.5 * iqr

    outliers = df[col] > upper_limit

    print(f"\n{col}")
    print("Upper outlier limit:", round(upper_limit, 2))
    print("Potential outlier records:", int(outliers.sum()))

# 4. Check daily arrival volumes
daily_volume = (
    df.dropna(subset=["Arrival"])
      .groupby(df["Arrival"].dt.date)
      .size()
      .sort_index()
)

if len(daily_volume) > 0:
    average_volume = daily_volume.mean()
    std_volume = daily_volume.std()

    # Flag unusually high-volume days
    threshold = average_volume + 2 * (
        std_volume if pd.notna(std_volume) else 0
    )

    unusual_days = daily_volume[daily_volume > threshold]

    print("\nDaily Arrival Volume")
    print("Average daily arrivals:", round(average_volume, 2))
    print("High-volume threshold:", round(threshold, 2))
    print("Potential high-volume days:", len(unusual_days))
    print(unusual_days)
else:
    print("\nNo valid arrival timestamps available.")

# 5. Assumptions
print("\nAssumptions:")
print("1. IQR identifies potential duration outliers, not confirmed errors.")
print("2. Days above the average plus two standard deviations are flagged.")
print("3. Negative durations are excluded from valid duration limits.")
print("4. Flagged records require operational review; values are not changed.")


Arrival_to_Triage_Min
Upper outlier limit: 131.0
Potential outlier records: 0

Arrival_to_Discharge_Hours
Upper outlier limit: 15.6
Potential outlier records: 0

Daily Arrival Volume
Average daily arrivals: 74.07
High-volume threshold: 91.99
Potential high-volume days: 8
Arrival
2026-01-02     94
2026-03-10    100
2026-03-21     98
2026-04-19     93
2026-04-22     93
2026-04-23     92
2026-08-03     96
2026-09-17    101
dtype: int64

Assumptions:
1. IQR identifies potential duration outliers, not confirmed errors.
2. Days above the average plus two standard deviations are flagged.
3. Negative durations are excluded from valid duration limits.
4. Flagged records require operational review; values are not changed.


In [18]:
## Scenario 15: Privacy-Safe Dashboard

MIN_GROUP_SIZE = 10

# Aggregate records by department and status
dashboard = (
    df.groupby(["Dept", "Status"], dropna=False)
      .size()
      .reset_index(name="Record_Count")
)

# Suppress groups smaller than the minimum size
dashboard["Displayed_Count"] = dashboard["Record_Count"].apply(
    lambda count: "Suppressed"
    if count < MIN_GROUP_SIZE
    else count
)

# Remove the original count to avoid exposing suppressed values
dashboard = dashboard.drop(columns=["Record_Count"])

print("Privacy-Safe Dashboard")
print(dashboard.to_string(index=False))

print("\nTotal dashboard groups:", len(dashboard))
print("Minimum group size:", MIN_GROUP_SIZE)
print("Suppressed groups:",
      (dashboard["Displayed_Count"] == "Suppressed").sum())

print("\nAssumptions:")
print("1. Groups with fewer than 10 records are suppressed.")
print("2. Only aggregate counts are displayed.")
print("3. Patient identifiers and clinical notes are not displayed.")
print("4. Small-group suppression alone does not guarantee full privacy.")

Privacy-Safe Dashboard
            Dept      Status  Displayed_Count
      CARDIOLOGY    ADMITTED             1578
      CARDIOLOGY   CANCELLED             1578
      CARDIOLOGY   COMPLETED             4704
      CARDIOLOGY  DISCHARGED             1586
      CARDIOLOGY IN PROGRESS             1474
      CARDIOLOGY     NO-SHOW             1537
       EMERGENCY    ADMITTED             1521
       EMERGENCY   CANCELLED             1563
       EMERGENCY   COMPLETED             4731
       EMERGENCY  DISCHARGED             1587
       EMERGENCY IN PROGRESS             1589
       EMERGENCY     NO-SHOW             1575
GENERAL MEDICINE    ADMITTED             1517
GENERAL MEDICINE   CANCELLED             1513
GENERAL MEDICINE   COMPLETED             4546
GENERAL MEDICINE  DISCHARGED             1555
GENERAL MEDICINE IN PROGRESS             1539
GENERAL MEDICINE     NO-SHOW             1588
             ICU    ADMITTED             1531
             ICU   CANCELLED             1578
           

In [19]:
#16. Compare service performance across facilities only after harmonizing definitions and timezones.
# Convert timestamps
for col in ["Arrival", "Triage", "Discharge"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Calculate service durations in minutes
df["Triage_Minutes"] = (
    df["Triage"] - df["Arrival"]
).dt.total_seconds() / 60

df["Discharge_Hours"] = (
    df["Discharge"] - df["Arrival"]
).dt.total_seconds() / 3600

# Exclude negative durations from performance averages
df.loc[df["Triage_Minutes"] < 0, "Triage_Minutes"] = pd.NA
df.loc[df["Discharge_Hours"] < 0, "Discharge_Hours"] = pd.NA

# Compare facilities using consistent metric definitions
facility_performance = df.groupby("Facility").agg(
    Total_Records=("Encounter_ID", "count"),
    Average_Triage_Minutes=("Triage_Minutes", "mean"),
    Average_Discharge_Hours=("Discharge_Hours", "mean")
).round(2)

print("Facility Performance Comparison")
print(facility_performance)

print("\nAssumptions and limitations:")
print("1. Triage time = Triage timestamp minus Arrival timestamp.")
print("2. Discharge time = Discharge timestamp minus Arrival timestamp.")
print("3. Negative durations are excluded from averages.")
print("4. Facility timezone information must be verified before comparison.")
print("5. Results are preliminary until timestamp definitions and timezones are confirmed.")

Facility Performance Comparison
                  Total_Records  Average_Triage_Minutes  \
Facility                                                  
CENTRAL HOSPITAL          20157                   46.81   
EAST HOSPITAL             20047                   46.64   
NORTH-HOSPITAL            19795                   47.11   
SOUTH-HOSPITAL            20172                   47.19   
WEST HOSPITAL             19829                   47.22   

                  Average_Discharge_Hours  
Facility                                   
CENTRAL HOSPITAL                     5.94  
EAST HOSPITAL                        5.94  
NORTH-HOSPITAL                       5.95  
SOUTH-HOSPITAL                       5.92  
WEST HOSPITAL                        5.97  

Assumptions and limitations:
1. Triage time = Triage timestamp minus Arrival timestamp.
2. Discharge time = Discharge timestamp minus Arrival timestamp.
3. Negative durations are excluded from averages.
4. Facility timezone information must be v

In [20]:
# Convert timestamps
df["Shift_Start"] = pd.to_datetime(df["Shift_Start"], errors="coerce", utc=True)
df["Shift_End"] = pd.to_datetime(df["Shift_End"], errors="coerce", utc=True)

# Correct overnight boundary only when end is earlier than start
overnight = (
    df["Shift_Start"].notna() &
    df["Shift_End"].notna() &
    (df["Shift_End"] < df["Shift_Start"])
)

df.loc[overnight, "Shift_End"] += pd.Timedelta(days=1)

# Recalculate duration for every record
df["Shift_Duration_Hours"] = (
    df["Shift_End"] - df["Shift_Start"]
).dt.total_seconds() / 3600

# Validate overnight shift records
night = df["Staff_Shift"].astype("string").str.strip().eq("22:00-06:00")

df["Shift_Review"] = (
    night &
    (
        df["Shift_Start"].isna() |
        df["Shift_End"].isna() |
        df["Shift_Start"].dt.hour.ne(22) |
        df["Shift_End"].dt.hour.ne(6) |
        df["Shift_Duration_Hours"].ne(8)
    )
)

# Summary for the entire dataset
print("Total dataset rows:", len(df))
print("Overnight shift rows:", night.sum())
print("Overnight rows needing review:", df["Shift_Review"].sum())
print("Valid overnight rows:", (night & ~df["Shift_Review"]).sum())

Total dataset rows: 100000
Overnight shift rows: 16452
Overnight rows needing review: 15966
Valid overnight rows: 486


In [21]:
df.to_csv("Hospital_Final_Data.csv", index=False)